# Figure 2: SHAP interpretation of expression-based mutation prediction

Reads the TCGA run outputs from `Runs/TCGA/output/cpm/` (per-target SHAP tables in `{cohort}/shap/` and cross-validation metrics in `{cohort}/cv/`) and writes panels to `Figures/Figure2/output/`.

SHAP values come from separate per-target XGBoost classifiers trained on the full cohort (`Runs/TCGA/src/run_tcga.py`), not from the multitask network.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

NOTEBOOK_DIR = Path.cwd().resolve()
FIGURES_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = FIGURES_DIR.parent
for path in (NOTEBOOK_DIR, FIGURES_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from constants import *
from figure2_helpers import (
    build_target_top_shap,
    load_beeswarm_table,
    load_mutation_metrics,
    load_shap_feature_summaries,
    pathway_long_table,
    plot_accuracy_vs_n_features,
    plot_beeswarm_from_table,
    plot_collapsed_pathway_dotplot,
    plot_directional_recurrence,
    plot_pathway_dotplot,
    plot_pathway_network,
    plot_target_feature_cancer_dotplot,
    recurrent_feature_table,
    save_table,
    shap_matrix_for_cohort,
    top_features_for_rows,
    truncated_cmap,
)

RESULTS_ROOT = PROJECT_ROOT / "Runs" / "TCGA" / "output" / "cpm"
FIGURE_ROOT = NOTEBOOK_DIR / "output"
SUPPLEMENT_ROOT = FIGURE_ROOT / "Supplemental"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
SUPPLEMENT_ROOT.mkdir(parents=True, exist_ok=True)

plt.rcParams["svg.fonttype"] = "none"

## Load SHAP summaries

In [ ]:
shap_long = load_shap_feature_summaries(RESULTS_ROOT, exclude_all=True)
shap_long = shap_long[shap_long['mean_shap'].abs() > 0.001]
print(f"Loaded {shap_long.shape[0]:,} SHAP feature-summary rows across {shap_long['cohort'].nunique()} cancer cohorts.")
print(f"Targets represented: {shap_long['target_gene'].nunique():,}")

## Panel a: AUPRC versus number of SHAP features

In [ ]:
# Figure 2a shows LIHC, ESCA, SKCM, LUAD, LGG and HNSC; all cancers are written for the supplement.
mutation_metrics = load_mutation_metrics(RESULTS_ROOT, exclude_all=True)
accuracy_shap_folder = FIGURE_ROOT / "accuracy_vs_n_features"
accuracy_shap_folder.mkdir(exist_ok=True)

annotate_threshold_map = {
    0.6: {"COAD", "STAD", "CHOL", "READ"},
    0.75: {"UCEC", "CHOL", "UCS", "DLBC"},
    0.5: {"SKCM"},
    0.4: {"LUAD"},
    0.3: {"BLCA", "CESC", "HNSC", "LUSC"},
}

accuracy_tables = []
for cancer in sorted(shap_long["cohort"].unique()):
    shap_df = shap_matrix_for_cohort(shap_long, cancer)
    if shap_df.empty or shap_df.shape[1] < 6:
        continue
    metric_sub = mutation_metrics.loc[mutation_metrics["cohort"].eq(cancer)].set_index("target_gene")
    if metric_sub.empty:
        continue
    annotate_thresh = 0.4
    for thresh, cancers in annotate_threshold_map.items():
        if cancer in cancers:
            annotate_thresh = thresh
            break
    plot_df = plot_accuracy_vs_n_features(
        shap_df=shap_df,
        acc=metric_sub["auprc"],
        prevalence=metric_sub["prevalence"],
        cancer=cancer,
        cmap=CMAP_ORANGE_GREEN,
        threshold=0,
        annotate_thresh=annotate_thresh,
        point_size=30,
        max_labels=250,
        figsize=(3, 2.4),
        save_path=accuracy_shap_folder / f"{cancer}_auprc_vs_n_features.pdf",
    )
    plot_df.insert(0, "cohort", cancer)
    accuracy_tables.append(plot_df.reset_index().rename(columns={"index": "target_gene"}))

accuracy_vs_n_features_df = pd.concat(accuracy_tables, ignore_index=True) if accuracy_tables else pd.DataFrame()
save_table(accuracy_vs_n_features_df, FIGURE_ROOT / "figure2a_auprc_vs_n_features_long.csv", index=False)
print(f"Wrote {len(accuracy_tables)} per-cancer accuracy-vs-SHAP-feature panels.")
display(accuracy_vs_n_features_df.sort_values("score", ascending=False).head(12))

## Panels b-c: TP53 recurrent SHAP features

In [ ]:
tp53_top = build_target_top_shap(shap_long, "TP53", top_n_per_cancer=50)
tp53_feature_summary = recurrent_feature_table(tp53_top, min_cancers=5, max_features=40)
save_table(tp53_feature_summary, FIGURE_ROOT / "figure2b_tp53_recurrent_feature_summary.csv", index=False)

print(f"TP53 SHAP rows after top-50-per-cancer filter: {tp53_top.shape[0]:,}")
print(f"TP53 cohorts: {tp53_top['cohort'].nunique()}")
display(tp53_feature_summary.head(15))

In [ ]:
tp53_cmap = truncated_cmap("magma_r", 0.1, 0.9)

tp53_dotplot_df = plot_target_feature_cancer_dotplot(
    tp53_top,
    target_gene="TP53",
    min_cancers=5,
    max_features=40,
    cmap=tp53_cmap,
    figsize=(4.5, 3.0),
    save_path=FIGURE_ROOT / "figure2b_tp53_recurrent_shap_dotplot.pdf",
)

tp53_direction_df = plot_directional_recurrence(
    tp53_top,
    target_gene="TP53",
    min_cancers=5,
    max_genes=20,
    figsize=(3.5, 3.0),
    save_path=FIGURE_ROOT / "figure2c_tp53_directional_recurrence.pdf",
)

display(tp53_direction_df)

## Panels d-e: curated pathway SHAP panels

In [ ]:
# Figure 2d: pathway_network_wnt_beta_catenin.pdf; Figure 2e: pathway_collapsed_nrf2.pdf.
# The remaining pathway panels are supplemental.
pathway_specs = {
        "wnt_beta_catenin": {
        "title": "WNT / beta-catenin / BRAF",
        "block_color": PURPLE,
        "rows": [
            ("ACC", "CTNNB1"),
            ("UCEC", "CTNNB1"),
            ("LIHC", "CTNNB1"),

            ("COAD", "APC"),
            ("READ", "APC"),
            ("STAD", "APC"),
            ("UCEC", "APC"),

            ("COAD", "RNF43"),
            ("STAD", "RNF43"),

            ("COAD", "BRAF"),
        ],
        "targets": ["CTNNB1", "APC", "RNF43", "BRAF"],
        "genes": [
            "NKD1", "TCF7", "DKK4", "GLUL", "MUCL1", "RNF43",
            "AXIN2", "ZNRF3", "FZD9", "NOTUM", "TDGF1", "LGR5",
            "RSPO4", "DKK1", "APCDD1", "CFTR", "MLH1", "SOX1", "ZIC2", "ZIC5",
            "HUNK", "EEPD1",
        ],
        "figsize_dotplot": (5.2, 3.8),
        "figsize_collapsed": (4.8, 1.8),
        "figsize_network": (4.0, 4.9),
        "legend_loc_network": "lower left",
        "legend_bbox_network": (0.12, 0.02),
    },
    "nrf2": {
        "title": "NRF2",
        "block_color": PURPLE,
        "rows": [
            ("LUAD", "KEAP1"),
            ("LUSC", "KEAP1"),
            ("ESCA", "NFE2L2"),
            ("BLCA", "NFE2L2"),
            ("LUSC", "NFE2L2"),
        ],
        "targets": ["KEAP1", "NFE2L2"],
        "genes": [
            "TRIM16L",
            "AKR1C2", "AKR1C4", "AKR1B10",
            "OSGIN1", "TXNRD1", "NQO1",
            "GCLC", "GCLM", "SLC7A11",
            "PGD", "G6PD", "TALDO1",
            "ABCC2", "EPHX1", "CES1",
            "CYP4F3", "CYP4F11",
            "PANX2", "SOST",
        ],
        "figsize_dotplot": (4.8, 2.7),
        "figsize_collapsed": (4.8, 1.5),
        "figsize_network": (3.4, 3.6),
        "legend_loc_network": "lower left",
        "legend_bbox_network": (0.25, 0.35),
    },
    "cic_sf3b1": {
        "title": "CIC / SF3B1",
        "block_color": TEAL,
        "rows": [("LGG", "CIC"), ("UVM", "SF3B1")],
        "targets": ["CIC", "SF3B1"],
        "genes": ["ETV4", "ETV1", "TRIM67", "DNAI4", "SRRM5", "SLC4A3", "PNKP", "LIG3"],
        "figsize_dotplot": (3, 1.8),
        "figsize_collapsed": (3.4, 1.5),
        "figsize_network": (3, 3.2),
        "legend_loc_network": "lower left",
        "legend_bbox_network": (0.15, 0.4),
    },
}

pathway_tables = []
network_tables = []
for slug, spec in pathway_specs.items():
    detail_df = plot_pathway_dotplot(
        shap_long,
        rows=spec["rows"],
        genes=spec["genes"],
        block_color=spec["block_color"],
        title=spec["title"],
        global_max=1,
        figsize=spec["figsize_dotplot"],
        save_path=FIGURE_ROOT / f"pathway_dotplot_{slug}.pdf",
    )
    detail_df.insert(0, "pathway", slug)
    pathway_tables.append(detail_df)

    plot_collapsed_pathway_dotplot(
        shap_long,
        targets=spec["targets"],
        genes=spec["genes"],
        block_color=spec["block_color"],
        title=spec["title"],
        figsize=spec["figsize_collapsed"],
        save_path=FIGURE_ROOT / f"pathway_collapsed_{slug}.pdf",
    )

    network_df = pathway_long_table(shap_long, rows=spec["rows"], genes=spec["genes"])
    ct_edges, tf_edges = plot_pathway_network(
        network_df,
        target_order=spec["targets"],
        candidate_features=spec["genes"],
        figsize=spec["figsize_network"],
        top_features_per_target=spec.get("top_features_per_target", 8),
        title=spec["title"],
        legend_loc=spec["legend_loc_network"],
        legend_bbox=spec["legend_bbox_network"],
        save_path=FIGURE_ROOT / f"pathway_network_{slug}.pdf",
    )
    ct_edges.insert(0, "pathway", slug)
    tf_edges.insert(0, "pathway", slug)
    ct_edges.insert(1, "edge_type", "cancer_target")
    tf_edges.insert(1, "edge_type", "target_feature")
    network_tables.append((ct_edges, tf_edges))

atrx_rows = [("LGG", "ATRX"), ("GBM", "ATRX")]
atrx_genes = top_features_for_rows(shap_long[shap_long["target_gene"].eq("ATRX")], atrx_rows, top_n=4)
plot_pathway_dotplot(
    shap_long,
    rows=atrx_rows,
    genes=atrx_genes,
    block_color=TEAL,
    title="ATRX",
    global_max=1,
    figsize=(1.8, 1.4),
    save_path=SUPPLEMENT_ROOT / "pathway_dotplot_atrx.pdf",
)

all_pathway_df = pd.concat(pathway_tables, ignore_index=True)
save_table(all_pathway_df, FIGURE_ROOT / "pathway_dotplots_long.csv", index=False)
all_network_edges = pd.concat([df for pair in network_tables for df in pair], ignore_index=True)
save_table(all_network_edges, FIGURE_ROOT / "pathway_network_edges_long.csv", index=False)
display(all_pathway_df.dropna(subset=["mean_abs_shap"]).sort_values("mean_abs_shap", ascending=False).head(20))

## Panel f: per-sample SHAP beeswarm panels

In [ ]:
# Figure 2f: shap_summary_LGG_CIC.pdf and shap_summary_UVM_SF3B1.pdf; the others are supplemental.
beeswarm_folder = FIGURE_ROOT / "shap_summary"
beeswarm_folder.mkdir(exist_ok=True)

cancer_gene_tuples = [
    ("ACC", "CTNNB1", 8),
    ("ACC", "MEN1", 8),
    ("ACC", "NF1", 8),
    ("ACC", "PRKAR1A", 10),
    ("BLCA", "FGFR3", 6),
    ("BLCA", "NFE2L2", 10),
    ("BLCA", "RB1", 6),
    ("BLCA", "TP53", 8),
    ("BRCA", "CDH1", 6),
    ("BRCA", "GATA3", 6),
    ("BRCA", "MAP3K1", 8),
    ("BRCA", "PIK3CA", 10),
    ("CESC", "ARID1A", 6),
    ("CESC", "GOLGB1", 10),
    ("CESC", "MUC17", 8),
    ("CESC", "PTEN", 8),
    ("COAD", "APC", 15),
    ("COAD", "BRAF", 8),
    ("COAD", "DCHS2", 6),
    ("COAD", "MUC5B", 6),
    ("COAD", "RNF43", 6),
    ("GBM", "ATRX", 6),
    ("GBM", "IDH1", 4),
    ("GBM", "RYR3", 8),
    ("HNSC", "CASP8", 12),
    ("HNSC", "HRAS", 8),
    ("HNSC", "NSD1", 8),
    ("LGG", "ATRX", 6),
    ("LGG", "CIC", 8),
    ("LGG", "IDH1", 6),
    ("LIHC", "CTNNB1", 6),
    ("LUAD", "EGFR", 8),
    ("LUAD", "KEAP1", 8),
    ("LUAD", "STK11", 8),
    ("LUSC", "NFE2L2", 4),
    ("MESO", "LATS2", 6),
    ("PAAD", "GNAS", 6),
    ("READ", "APC", 8),
    ("READ", "SOX9", 4),
    ("UVM", "EIF1AX", 8),
    ("UVM", "SF3B1", 8),
]

CMAP_ORANGE_PURPLE_VIVID = LinearSegmentedColormap.from_list(
    "orange_purple_vivid",
    ["#5A49C6", "#FFF3DB", "#E12A5F"],
)

beeswarm_manifest = []
missing_beeswarms = []
for cancer, gene, n_features in cancer_gene_tuples:
    try:
        beeswarm_df = load_beeswarm_table(RESULTS_ROOT, cancer, gene)
    except FileNotFoundError as exc:
        missing_beeswarms.append({"cohort": cancer, "target_gene": gene, "reason": str(exc)})
        continue

    plot_df = plot_beeswarm_from_table(
        beeswarm_df,
        cancer=cancer,
        target_gene=gene,
        max_display=n_features,
        cmap=CMAP_ORANGE_PURPLE_VIVID,
        figsize=(2.8, 0.2 * n_features + 0.8),
        title=f"{cancer} - {gene} SHAP",
        save_path=beeswarm_folder / f"shap_summary_{cancer}_{gene}.pdf",
    )
    beeswarm_manifest.append(
        {
            "cohort": cancer,
            "target_gene": gene,
            "n_features_requested": n_features,
            "n_features_plotted": int(plot_df["feature"].nunique()),
            "n_samples": int(beeswarm_df.shape[0]),
            "source_file": f"{cancer}/shap/{gene}/sample_shap_top_features.csv.gz",
        }
    )

beeswarm_manifest_df = pd.DataFrame(beeswarm_manifest)
save_table(beeswarm_manifest_df, FIGURE_ROOT / "beeswarm_manifest.csv", index=False)
if missing_beeswarms:
    display(pd.DataFrame(missing_beeswarms))
print(f"Saved {len(beeswarm_manifest_df)} beeswarm panels to {beeswarm_folder.relative_to(PROJECT_ROOT)}")
display(beeswarm_manifest_df.head())

## Output inventory

In [ ]:
for folder in [FIGURE_ROOT, SUPPLEMENT_ROOT, FIGURE_ROOT / "shap_summary"]:
    print(folder.relative_to(PROJECT_ROOT))
    for path in sorted(folder.glob("*")):
        if path.is_file():
            print("  ", path.name)